# 8장 RAG (풀이)

검증용 풀이 노트북입니다. 공유하지 않습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(1장의 공개 LLM, 임베딩 모델)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [1]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

사용하는 API: anthropic


In [2]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install sentence-transformers rank-bm25 pypdf
# 8장과 관통 프로젝트 3단계에서 씁니다. 문서를 읽고, 청크로 나누고, 검색하는 함수는 같은 폴더의 corpus.py에 있습니다
import csv
import random
import re

import numpy as np

import corpus
import embed

with open(f'{DATA}/문의.csv', encoding='utf-8') as file:
    inquiries = list(csv.DictReader(file))        # 문의 60건. 관통 프로젝트 1, 2단계의 코드가 씁니다
LABELS = ['설비 이상', '품질', '안전', '문서와 데이터 요청']

# 100만 토큰당 단가(달러). 5장과 같습니다(llm.py가 쓰는 Anthropic 모델, 2026-11 확인)
PRICES = {'소형': {'input': 0.10, 'output': 0.50}, '기본': {'input': 2.00, 'output': 10.00},
          '대형': {'input': 4.00, 'output': 20.00}}


def cost(reply, tier):
    """응답 하나의 요금(달러). 입력 토큰과 출력 토큰(thinking 포함)에 각각의 단가를 곱해 더합니다"""
    price = PRICES[tier]
    return (reply.input_tokens * price['input'] + reply.output_tokens * price['output']) / 1_000_000


catalog = corpus.catalog()                               # {문서 id: 메타데이터}. 문서 120건
questions = {item['id']: item for item in corpus.evalset()}   # 평가 질문 62개
chunks, failed = corpus.load_chunks()                    # 문서마다 파싱하고 절 단위 청크로 나눕니다(corpus.py)
searcher = corpus.Search(chunks)                         # 청크마다 임베딩과 BM25 인덱스를 만듭니다. 1~2분 걸립니다


def is_valid(chunk):
    """메타데이터 필터. 폐기된 구판의 청크를 검색 대상에서 뺍니다"""
    return chunk['status'] == '유효'


RAG_PROMPT = """아래 [검색 결과]만 근거로 질문에 답하세요.
- 문장마다 근거가 된 청크의 번호를 [SOP-WB-001#3]처럼 적습니다.
- 검색 결과에 답이 없으면 "검색한 문서에서 찾지 못했습니다"라고만 답합니다.
- 검색 결과끼리 내용이 다르면 날짜가 늦은 문서를 따르고, 내용이 다르다는 것을 함께 적습니다.

[검색 결과]
{context}

[질문]
{question}"""


def rag_prompt(question, hits, with_meta=True):
    """검색한 청크를 번호와 함께 RAG_PROMPT에 넣습니다. with_meta가 참이면 청크마다 상태, 개정, 날짜를 붙입니다"""
    blocks = []
    for hit in hits:
        meta = f" (상태 {hit['status']}, 개정 {hit['revision']}, {hit['date']})" if with_meta else ''
        blocks.append(f"[{hit['chunk_id']}]{meta}\n{hit['text']}")
    return RAG_PROMPT.format(context='\n\n'.join(blocks), question=question)


print(f'문서 {len(catalog)}건, 청크 {len(chunks)}개, 글을 뽑지 못한 문서 {failed}')


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


임베딩 모델: intfloat/multilingual-e5-small (PC에 저장된 파일을 씁니다)


문서 120건, 청크 487개, 글을 뽑지 못한 문서 ['EHS-010']
소형 모델: claude-haiku-5-5
기본 모델: claude-sonnet-5-5
대형 모델: claude-opus-5-5


준비 완료


오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 8-1. 검색 없이 묻기와 검색 결과를 넣어 묻기

교재의 절: "RAG의 구조"

사내 문서에만 답이 있는 평가 질문 두 개를 검색 없이 한 번, 유효한 문서에서 검색한 청크 4개를 넣어 한 번 보냅니다. 검색 없이 물었을 때 모델이 무엇이라고 답하는지, 검색 결과를 넣었을 때 정답 값과 출처가 나오는지 봅니다. 이 셀은 API를 4번 호출합니다.

In [3]:
for qid in ['E06', 'E24']:
    question = questions[qid]['question']
    alone = llm.ask(question, model='소형', max_tokens=2000)              # 검색 없이 묻습니다
    hits = searcher.search(question, k=4, where=is_valid)                 # 유효한 문서의 청크 상위 4개
    grounded = llm.ask(rag_prompt(question, hits), model='소형', max_tokens=2000)
    print(f"{qid} {question} (정답: {questions[qid]['answer']})")
    print('  검색 없이:', alone.text.strip().replace('\n', ' ')[:180])
    print('  검색한 청크:', [hit['chunk_id'] for hit in hits])
    print(f'  검색 결과를 넣어(입력 {grounded.input_tokens:,}토큰):', grounded.text.strip().replace('\n', ' ')[:180])

E06 패키지 테스트에서 로트 수율이 몇 % 미만이면 로트를 홀드하나요? (정답: 97% 미만이면 홀드한다.)
  검색 없이: 패키지 테스트의 로트 홀드 기준 수율은 **정해진 업계 표준 수치가 없고, 회사·제품·공정마다 다릅니다.** 어떤 회사에서는 90% 미만, 어떤 곳에서는 95% 미만을 기준으로 쓰기도 하므로, 임의의 숫자를 답으로 드리기는 어렵습니다.  기준은 보통 아래 문서나 자료에서 확인할 수 있습니다.  - **사내 품질관리 기준서
  검색한 청크: ['SOP-PT-001#4', 'QIR-2026-015#2', 'EDU-006#3', 'MTG-20260601#3']
  검색 결과를 넣어(입력 1,091토큰): 로트 수율이 **97% 미만**이면 로트를 홀드하고 테스트기술팀과 품질보증팀에 알립니다 [SOP-PT-001#4].


E24 D-03 블레이드 때문에 생긴 칩핑 불량 때 블레이드의 누적 절단 거리는 얼마였나요? (정답: 11.5km.)
  검색 없이: 해당 불량 건의 D-03 블레이드 누적 절단 거리는 제가 알 수 없습니다. 제게는 공장의 설비 로그나 품질 기록에 대한 접근 권한이 없고, 이 대화에서 공유된 자료도 없습니다.  다음 자료에서 확인하실 수 있습니다.  - **블레이드 사용 이력 / 수명 관리 대장**: 장착일, 교체일, 누적 컷 수, 누적 절단 길이 등이
  검색한 청크: ['QIR-2025-044#3', 'MAN-DS-001#3', 'QIR-2025-044#5', 'MTG-20260119#3']
  검색 결과를 넣어(입력 1,039토큰): D-03 블레이드의 누적 절단 거리는 **11.5km**였습니다[QIR-2025-044#3].  당시 교체 기준은 12km였으므로 교체 기준에는 이르지 않았지만, BG256은 웨이퍼가 150μm로 얇아 마모된 블레이드의 영향을 더 크게 받았습니다[QIR-2025-044#3].  참고로 검색 결과 중 더 최신 문서인 MAN


## 예제 8-2. 형식마다 다른 파싱

교재의 절: "문서 파싱"

HTML 문서 하나와 PDF 문서 두 개(표가 든 매뉴얼, 스캔 문서)를 단순한 방법(HTML은 태그만 지우기, PDF는 모든 쪽의 글자 층)으로 뽑은 글과 `corpus.read_text()`가 뽑은 본문을 비교합니다. 글자 수가 얼마나 줄었는지, 본문에서 뺀 줄이 무엇인지, 스캔 문서에서 몇 글자가 나오는지 봅니다. API를 호출하지 않습니다.

In [4]:
from pypdf import PdfReader

for doc_id in ['EDU-003', 'MAN-WB-001', 'EHS-010']:
    info = catalog[doc_id]
    path = f"{DATA}/corpus/{info['file']}"
    if info['format'] == 'html':
        raw = re.sub(r'<[^>]+>', '', open(path, encoding='utf-8').read())     # 태그만 지운 글
    else:
        raw = '\n'.join(page.extract_text() for page in PdfReader(path).pages)  # 모든 쪽의 글자 층
    body = corpus.read_text(doc_id)
    removed = [line.strip() for line in raw.splitlines() if line.strip() and line.strip() not in body]
    print(f"{doc_id}({info['format']}): 단순 추출 {len(raw.strip()):,}자, 본문 {len(body):,}자")
    print('  본문에서 뺀 줄:', removed[:4])

EDU-003(html): 단순 추출 896자, 본문 667자
  본문에서 뺀 줄: ['신입 사원 라인 투입 교육 과정 - 사내 위키', 'body { font-family: sans-serif; max-width: 860px; margin: auto; } table { border-collapse: collapse; } td, th { border: 1px solid #999; padding: 4px 8px; }', '사내 위키 | 홈 | 최근 바뀜 | 문서 검색 | 로그아웃', '홈 &gt; 교육 자료 &gt; EDU-003']
MAN-WB-001(pdf): 단순 추출 1,266자, 본문 1,149자
  본문에서 뺀 줄: ['MAN-WB-001  개정 3  조립기술팀  사내 한정', '1/2  출력본은 최신 개정본이 아닐 수 있습니다', 'MAN-WB-001  개정 3  조립기술팀  사내 한정', '2/2  출력본은 최신 개정본이 아닐 수 있습니다']
EHS-010(pdf): 단순 추출 0자, 본문 0자
  본문에서 뺀 줄: []


## 예제 8-3. 512토큰 뒤의 내용은 벡터에 없다

교재의 절: "청크 분할"

가장 긴 문서 EDU-001(후공정 개요)의 토큰 수를 세고, 문서 전체로 만든 벡터와 앞부분만으로 만든 벡터를 비교합니다. 두 벡터의 유사도가 1.0이면 뒷부분은 벡터에 아무 영향을 주지 않은 것입니다. 뒷부분의 마킹 절을 묻는 질문이 문서 전체 벡터와 마킹 절 청크 가운데 어느 쪽에 더 가까운지도 봅니다. API를 호출하지 않습니다.

In [5]:
text = corpus.read_text('EDU-001')
tokenizer = embed.load().tokenizer
cut = text.index('## 7.')                                         # 1~6절까지만 남긴 앞부분
print(f"EDU-001 전체 {len(tokenizer(text)['input_ids']):,}토큰, 1~6절 {len(tokenizer(text[:cut])['input_ids']):,}토큰"
      f"(최대 입력 길이 {embed.load().max_seq_length}토큰)")
whole, front = embed.passages([text, text[:cut]])
print(f'문서 전체 벡터와 1~6절 벡터의 유사도: {whole @ front:.4f}')
question = embed.query('패키지 윗면에 레이저로 무엇을 새기나요?')
marking = next(chunk for chunk in chunks if chunk['doc_id'] == 'EDU-001' and chunk['section'].startswith('9.'))
print(f"질문과 문서 전체 벡터: {whole @ question:.3f}, 질문과 마킹 절 청크: {embed.passages([marking['text']])[0] @ question:.3f}")
print('마킹 절 청크:', marking['text'][:80])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1285 > 512). Running this sequence through the model will result in indexing errors


EDU-001 전체 1,285토큰, 1~6절 859토큰(최대 입력 길이 512토큰)


문서 전체 벡터와 1~6절 벡터의 유사도: 1.0000


질문과 문서 전체 벡터: 0.828, 질문과 마킹 절 청크: 0.878
마킹 절 청크: EDU-001 후공정 개요 > 9. 마킹
패키지 윗면에 제품 이름, 로트 번호, 날짜 코드를 새긴다. 주로 레이저로 표면을 얇게 태워 글자를 만


## 예제 8-4. 구판이 섞였을 때

교재의 절: "RAG 프롬프트"

구판이 따로 남아 있는 문서의 기준값을 묻는 질문 두 개(E28 컴파운드 해동 시간, E31 리드 코플래너리티)를 세 가지 방법으로 보냅니다. (가) 구판까지 검색하고 청크 글만 넣기, (나) 구판까지 검색하고 청크에 상태, 개정, 날짜를 붙여 넣기, (다) 유효한 문서만 검색하기. 응답에 신판의 값과 구판의 값 가운데 무엇이 나오는지 봅니다. 이 셀은 API를 6번 호출합니다.

In [6]:
values = {'E28': ('6시간', '4시간'), 'E31': ('0.08mm', '0.10mm')}       # (신판의 값, 구판의 값)
for qid, (new, old) in values.items():
    question = questions[qid]['question']
    everything = searcher.search(question, k=4)                         # 구판도 검색 대상에 둡니다
    print(f'{qid} {question}\n  구판까지 검색한 청크: {[hit["chunk_id"] for hit in everything]}')
    for name, hits, with_meta in [('(가) 구판 포함, 메타데이터 없음', everything, False),
                                  ('(나) 구판 포함, 메타데이터 있음', everything, True),
                                  ('(다) 유효한 문서만 검색', searcher.search(question, k=4, where=is_valid), True)]:
        reply = llm.ask(rag_prompt(question, hits, with_meta), model='소형', max_tokens=2000)
        print(f'  {name}: 신판 값 {new in reply.text}, 구판 값 {old in reply.text}')
        print('    ', reply.text.strip().replace('\n', ' ')[:150])

E28 몰딩 컴파운드는 냉장 창고에서 꺼내 최소 몇 시간 해동하나요?
  구판까지 검색한 청크: ['SOP-MD-002#4', 'SOP-MD-002_R1#4', 'SOP-MD-002#3', 'SOP-MD-002_R1#3']

  (가) 구판 포함, 메타데이터 없음: 신판 값 True, 구판 값 True
     몰딩 컴파운드는 냉장 창고에서 꺼낸 뒤 포장을 뜯지 않은 채 상온에서 **최소 4시간** 해동합니다 [SOP-MD-002_R1#3].  검색 결과에 따라 내용이 다릅니다. 원본 문서 SOP-MD-002에서는 최소 6시간으로 되어 있습니다 [SOP-MD-002#3]. 검


  (나) 구판 포함, 메타데이터 있음: 신판 값 True, 구판 값 True
     몰딩 컴파운드는 냉장 창고에서 꺼낸 뒤 포장을 뜯지 않은 채 상온에서 **최소 6시간** 해동합니다 [SOP-MD-002#3].  검색 결과에는 같은 항목의 이전 개정본(SOP-MD-002_R1, 2024-03-11, 상태 폐기)이 있으며, 여기에는 최소 4시간으로 적


  (다) 유효한 문서만 검색: 신판 값 True, 구판 값 False
     몰딩 컴파운드는 냉장 창고에서 꺼낸 뒤 포장을 뜯지 않은 채 상온에서 최소 6시간 해동합니다 [SOP-MD-002#3]. 포장을 일찍 뜯으면 차가운 정제 표면에 수분이 맺혀 기포와 미충전의 원인이 되므로 해동 중에는 포장을 열지 않아야 합니다 [SOP-MD-002#3]


E31 QX48의 리드 코플래너리티는 몇 mm를 넘으면 불량인가요?
  구판까지 검색한 청크: ['MTG-20260907#3', 'SOP-VI-001_R3#3', 'SOP-VI-001#3', 'QIR-2026-002#3']


  (가) 구판 포함, 메타데이터 없음: 신판 값 True, 구판 값 True
     QX48의 리드 코플래너리티는 **0.10mm 또는 0.08mm를 초과하면 불량**입니다. 검색 결과의 두 문서가 기준을 다르게 적고 있습니다.  - SOP-VI-001 R3 개정본은 0.10mm 초과를 불량으로 봅니다 [SOP-VI-001_R3#3]. - 다른 SOP


  (나) 구판 포함, 메타데이터 있음: 신판 값 True, 구판 값 True
     QX48의 리드 코플래너리티는 **0.08mm를 초과하면 불량**입니다 [SOP-VI-001#3].  검색 결과에 기준이 서로 다른 문서가 있습니다. 개정 3(2025-06-16, 상태 폐기) 문서는 0.10mm 초과를 불량으로 보았지만 [SOP-VI-001_R3#3]


  (다) 유효한 문서만 검색: 신판 값 True, 구판 값 False
     QX48의 리드 코플래너리티는 **0.08mm를 초과하면 불량**입니다 [SOP-VI-001#3]. 리드 코플래너리티는 패키지를 평면에 놓았을 때 가장 높이 뜬 리드와 평면 사이의 거리입니다 [SOP-VI-001#3].  회의록에도 고객사 B가 QX48의 리드 코플래너


## 실습 8-1. 청크에 제목이 없으면

교재의 절: "개념 확인 실습"

품질 이슈 보고서 18건의 청크만 모아, 청크 첫 줄(문서 번호, 제목, 절 제목)이 있을 때와 뺐을 때 각각 벡터 인덱스를 만듭니다. 질문 "QIR-2026-014의 원인은 무엇이었나요?"로 검색했을 때 QIR-2026-014의 "원인" 절 청크가 몇 위에 오는지 두 경우 모두 먼저 `my_guess`에 적고, `bare`에 첫 줄을 뺀 청크 글의 목록을 만든 뒤 실행합니다. 두 순위의 차이를 청크 글에 무엇이 들어 있는지로 설명하면 끝납니다. API를 호출하지 않습니다.

In [7]:
qir = [chunk for chunk in chunks if chunk['type'] == '품질 이슈 보고서']
target = next(index for index, chunk in enumerate(qir) if chunk['doc_id'] == 'QIR-2026-014' and chunk['section'] == '원인')
bare = [chunk['text'].split('\n', 1)[1] for chunk in qir]
my_guess = {'제목 있음': 1, '제목 없음': 20}

question = 'QIR-2026-014의 원인은 무엇이었나요?'
for name, texts in [('제목 있음', [chunk['text'] for chunk in qir]), ('제목 없음', bare)]:
    scores = corpus.VectorIndex(texts).scores(question)
    print(f'{name}: 원인 절 청크 {list(np.argsort(-scores)).index(target) + 1}위(청크 {len(texts)}개 중), 예측 {my_guess[name]}')
print('제목 없는 원인 절 청크:', bare[target] if bare else '')

제목 있음: 원인 절 청크 6위(청크 87개 중), 예측 1


제목 없음: 원인 절 청크 73위(청크 87개 중), 예측 20
제목 없는 원인 절 청크: 리드 프레임을 플라즈마 세정한 뒤 9시간이 지나 본딩했다. 당시에는 세정한 매거진을 언제까지 써야 하는지 정한 기준이 없어, 작업자가 세정 여부를 판단했다. 시간이 지나면서 패드 표면에 유기물이 다시 쌓였다.
